In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import shap

d:\cproject\b-tech-course-project\synthetic data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
df = pd.read_csv("synthetic_subsidy_cylinders1.csv")


In [60]:
#scaler = StandardScaler().fit_transform(df.loc[:,df.columns!='Fraud'])
X=df.drop(columns=["Fraud","ID"])
print(X.columns)

model = IsolationForest(contamination=0.02, random_state=42)
model.fit(X)
X["anomaly_score"]=model.predict(X)
df['Fraud'] = X['anomaly_score'].apply(lambda x: 1 if x == -1 else 0)
print(df.loc[(df["Fraud"]>0) | (df["Fraud"]<0)])







Index(['Age', 'Gender', 'Marital_Status', 'Household_Size', 'Governorate',
       'Salary', 'Degree_Level', 'Employment_Status', 'Primary_Income_Source',
       'Has_Other_Social_Benefits', 'Assets_Value', 'Liabilities_Value',
       'Number_of_Children', 'Total_Spouse_Income', 'Working_Children_Count',
       'Total_Children_Income', 'Total_Household_Income', 'Vehicle_Ownership',
       'Vehicle_Count', 'Cylinder_Count', 'Vehicle_Age_Years', 'Fuel_Type',
       'Expected_Fuel_Consumption_L', 'Average_Fuel_Consumption_L',
       'Fuel_Deviation_L', 'Fuel_Deviation_Ratio', 'Previous_Subsidy_Received',
       'Previous_Subsidy_Amount', 'Late_or_Missed_Renewals',
       'Applications_Last_12_Months', 'Eligible'],
      dtype='object')
      Age  Gender  Marital_Status  Household_Size  Governorate   Salary  \
64   52.0       1               0               3            1   102.86   
202  61.0       0               1               5            1    80.00   
290  22.0       0               2

In [66]:
import pickle as pk
with open("frmodel.pk","wb") as file:
    pk.dump(model,file)

In [63]:
explainer = shap.Explainer(model)
shap_values = explainer(X)

# 3. Visualize for a specific "Under Review" applicant
# Let's say index 5 was flagged as fraud

# Isolate the specific row first
row_to_explain = df.loc[df["ID"] == 12947784].drop(columns=["ID", "Fraud"])

# Call the explainer on that row
row_shap_values = explainer(row_to_explain,check_additivity=False)

column_names = df.columns.drop(["ID", "Fraud"])
adf=pd.DataFrame(columns=column_names ,data=row_shap_values.values)
print(adf,adf.min().idxmin())
print(adf.iloc[0].sort_values(ascending=True).head(3).index.tolist())

        Age    Gender  Marital_Status  Household_Size  Governorate    Salary  \
0 -0.173351  0.018758       -0.381722       -0.009917    -0.108289 -0.111388   

   Degree_Level  Employment_Status  Primary_Income_Source  \
0       0.07355          -0.035145              -0.036434   

   Has_Other_Social_Benefits  ...  Fuel_Type  Expected_Fuel_Consumption_L  \
0                  -0.275914  ...  -0.174202                    -0.129264   

   Average_Fuel_Consumption_L  Fuel_Deviation_L  Fuel_Deviation_Ratio  \
0                   -0.086516          0.055451             -0.012025   

   Previous_Subsidy_Received  Previous_Subsidy_Amount  \
0                  -0.205586                -0.060977   

   Late_or_Missed_Renewals  Applications_Last_12_Months  Eligible  
0                  0.09237                     0.043165 -0.344767  

[1 rows x 31 columns] Number_of_Children
['Number_of_Children', 'Vehicle_Ownership', 'Marital_Status']


In [54]:
print(df.columns)
re = df.loc[df["ID"] == 14670588]
print(re.columns)
pr=model.predict(re.drop(columns=["Fraud"]))
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(df.loc[df["ID"] == 14670588].drop(columns=["Fraud"]))
print('5')
column_names = df.columns.drop(["Fraud"])
eligire=pd.DataFrame(columns=column_names ,data=shap_values)
print(eligire,eligire.min().idxmin(),eligire[eligire.min().idxmin()])

Index(['Age', 'Gender', 'Marital_Status', 'Household_Size', 'Governorate',
       'Salary', 'Degree_Level', 'Employment_Status', 'Primary_Income_Source',
       'Has_Other_Social_Benefits', 'Assets_Value', 'Liabilities_Value',
       'Number_of_Children', 'Total_Spouse_Income', 'Working_Children_Count',
       'Total_Children_Income', 'Total_Household_Income', 'Vehicle_Ownership',
       'Vehicle_Count', 'Cylinder_Count', 'Vehicle_Age_Years', 'Fuel_Type',
       'Expected_Fuel_Consumption_L', 'Average_Fuel_Consumption_L',
       'Fuel_Deviation_L', 'Fuel_Deviation_Ratio', 'Previous_Subsidy_Received',
       'Previous_Subsidy_Amount', 'Late_or_Missed_Renewals',
       'Applications_Last_12_Months', 'Fraud', 'Eligible', 'ID'],
      dtype='object')
Index(['Age', 'Gender', 'Marital_Status', 'Household_Size', 'Governorate',
       'Salary', 'Degree_Level', 'Employment_Status', 'Primary_Income_Source',
       'Has_Other_Social_Benefits', 'Assets_Value', 'Liabilities_Value',
       'Number_o

In [65]:
df.to_csv("synthetic_subsidy_cylinders2.csv",index=False)